# LeetCode #180 — Consecutive Numbers: tự nối bảng (self-join), `LAG/LEAD` và "đảo liên tiếp"

**Mức độ:** Medium · **Chủ đề:** Database (SQL), Pandas

Skeleton trên LeetCode:

```sql
-- MySQL
# Write your MySQL query statement below
```

```python
# Pandas
import pandas as pd

def consecutive_numbers(logs: pd.DataFrame) -> pd.DataFrame:
```

Mục tiêu của notebook:

1. **Tự nối bảng** (self-join): ghép mỗi dòng với dòng **kế tiếp** và dòng **kế tiếp nữa** bằng điều kiện trên `id`.
2. Hàm cửa sổ **`LAG` / `LEAD`**: nhìn sang dòng trước/sau mà không cần nối.
3. Kỹ thuật **"đảo liên tiếp"** (gaps and islands): hiệu hai `ROW_NUMBER()` không đổi trong một đoạn liên tiếp → `GROUP BY ... HAVING COUNT(*) >= 3`.
4. `DISTINCT` để mỗi số chỉ xuất hiện một lần trong kết quả.

# 1. Bài toán là gì?

Bảng `Logs`:

```text
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| num         | varchar |
+-------------+---------+
id là khóa chính, tự tăng bắt đầu từ 1.
```

Tìm mọi số xuất hiện **ít nhất ba lần liên tiếp** (theo thứ tự `id`). Kết quả: cột `ConsecutiveNums`, thứ tự tùy ý, mỗi số một lần.

---

## Ví dụ

```text
Logs                     Kết quả
| id | num |             | ConsecutiveNums |
| 1  | 1   |             | 1               |
| 2  | 1   |
| 3  | 1   |             (1 xuất hiện ở id 1, 2, 3 liên tiếp;
| 4  | 2   |              2 xuất hiện hai lần liên tiếp ở id 6, 7 — chưa đủ 3)
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. SQL cơ bản — truy vấn bảng dữ liệu
2. Hàm cửa sổ (window function) trong SQL
3. Pandas DataFrame — SQL bằng Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 SQL cơ bản — truy vấn bảng dữ liệu

### Bảng, hàng, cột

Cơ sở dữ liệu quan hệ lưu dữ liệu trong các **bảng**. Mỗi **hàng** là một bản ghi, mỗi **cột** là một thuộc tính có kiểu (số, chuỗi…). Ô có thể **rỗng** — giá trị đặc biệt `NULL` ("không biết / không có").

```text
Employee
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
+----+-------+--------+--------------+
```

Trong notebook ta chạy SQL thật bằng **`sqlite3`** (có sẵn trong Python, chạy trong bộ nhớ). LeetCode dùng MySQL — cú pháp các câu cơ bản giống nhau, chỗ khác biệt sẽ được ghi chú.

In [1]:
import sqlite3

def sql_demo(query, params=()):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER)")
    con.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)",
                    [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                     (4, "Sam", 60000, 2), (5, "Max", 90000, 1)])
    con.execute("CREATE TABLE Department (id INTEGER, name TEXT)")
    con.executemany("INSERT INTO Department VALUES (?, ?)", [(1, "IT"), (2, "Sales")])
    cur = con.execute(query, params)
    cols = [d[0] for d in cur.description]
    out = cur.fetchall()
    con.close()
    print(cols)
    for r in out:
        print("  ", r)

sql_demo("SELECT * FROM Employee")

['id', 'name', 'salary', 'departmentId']
   (1, 'Joe', 70000, 1)
   (2, 'Jim', 90000, 1)
   (3, 'Henry', 80000, 2)
   (4, 'Sam', 60000, 2)
   (5, 'Max', 90000, 1)


### Thứ tự **viết** và thứ tự **thực hiện** của một câu SELECT

```text
Viết:        SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...
Thực hiện:   FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Hiểu thứ tự thực hiện giải thích nhiều lỗi: `WHERE` chạy **trước** khi gom nhóm nên không dùng được hàm gộp (`COUNT`, `MAX`…) trong `WHERE` — phải dùng `HAVING`.

### Lọc, sắp xếp, giới hạn, bỏ trùng

In [2]:
print("Lương > 65000, giảm dần, lấy 2 hàng:")
sql_demo("SELECT name, salary FROM Employee WHERE salary > 65000 ORDER BY salary DESC LIMIT 2")
print("Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):")
sql_demo("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

Lương > 65000, giảm dần, lấy 2 hàng:
['name', 'salary']
   ('Jim', 90000)
   ('Max', 90000)
Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):
['salary']
   (80000,)


### Hàm gộp và GROUP BY

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` gộp nhiều hàng thành một giá trị. `GROUP BY cột` chia bảng thành nhóm theo cột đó, gộp riêng từng nhóm. `HAVING` lọc **sau** khi gộp.

In [3]:
sql_demo("""
SELECT departmentId, COUNT(*) AS n, MAX(salary) AS top
FROM Employee
GROUP BY departmentId
HAVING COUNT(*) >= 2
""")

['departmentId', 'n', 'top']
   (1, 3, 90000)
   (2, 2, 80000)


### NULL: "không có giá trị"

- Hàm gộp trên **tập rỗng** (trừ `COUNT`) trả về `NULL` — đây là mẹo để trả về `NULL` khi "không có đáp án".
- So sánh với `NULL` bằng `=` luôn **không đúng** → phải dùng `IS NULL`.
- `IFNULL(x, d)` / `COALESCE(x, d)` thay `NULL` bằng giá trị mặc định.

In [4]:
sql_demo("SELECT MAX(salary) AS m FROM Employee WHERE salary > 1000000")
sql_demo("SELECT NULL = NULL AS bang_nhau, NULL IS NULL AS la_null, COALESCE(NULL, 0) AS mac_dinh")

['m']
   (None,)
['bang_nhau', 'la_null', 'mac_dinh']
   (None, 1, 0)


### Truy vấn con (subquery)

Một câu `SELECT` có thể nằm **bên trong** câu khác: trả về một giá trị (dùng trong `WHERE x > (...)`), một danh sách (dùng với `IN`), hoặc một bảng tạm (đặt trong `FROM`).

In [5]:
print("Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:")
sql_demo("SELECT MAX(salary) AS second FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:
['second']
   (80000,)


### JOIN: ghép hai bảng

`A JOIN B ON điều_kiện` tạo các cặp hàng (a, b) thỏa điều kiện.

- **INNER JOIN** (mặc định): chỉ giữ cặp khớp.
- **LEFT JOIN**: giữ **mọi** hàng của A; hàng không khớp thì cột của B là `NULL`.
- **Self-join**: ghép bảng với **chính nó** (đặt hai bí danh khác nhau) — để so sánh các hàng trong cùng bảng.

In [6]:
sql_demo("""
SELECT d.name AS department, e.name AS employee, e.salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
ORDER BY d.name, e.salary DESC
""")
print("Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau")
sql_demo("""
SELECT a.name, b.name FROM Employee a JOIN Employee b
ON a.departmentId = b.departmentId AND a.salary < b.salary
""")

['department', 'employee', 'salary']
   ('IT', 'Jim', 90000)
   ('IT', 'Max', 90000)
   ('IT', 'Joe', 70000)
   ('Sales', 'Henry', 80000)
   ('Sales', 'Sam', 60000)
Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau
['name', 'name']
   ('Joe', 'Jim')
   ('Joe', 'Max')
   ('Sam', 'Henry')


### Lỗi hay gặp

- Quên `DISTINCT` khi hỏi "giá trị lớn thứ k **khác nhau**".
- Dùng hàm gộp trong `WHERE` (phải dùng `HAVING` hoặc truy vấn con).
- `LIMIT 1 OFFSET k` trên bảng **không đủ** hàng trả về **không hàng nào** (không phải `NULL`) — bọc trong truy vấn con hoặc dùng `MAX` để có `NULL`.
- So sánh `= NULL` thay vì `IS NULL`.
- Trong MySQL, `LIMIT` không nhận biểu thức (`LIMIT N-1` lỗi) → phải tính trước (`SET N = N - 1`).

## 2.2 Hàm cửa sổ (window function) trong SQL

### Hàm gộp vs hàm cửa sổ

- **Hàm gộp + GROUP BY**: mỗi nhóm **co lại** thành một hàng.
- **Hàm cửa sổ** `f(...) OVER (PARTITION BY ... ORDER BY ...)`: **giữ nguyên** mọi hàng, mỗi hàng nhận thêm một giá trị tính trên "cửa sổ" các hàng liên quan.

```text
PARTITION BY departmentId   → chia thành các nhóm (giống GROUP BY nhưng không co lại)
ORDER BY salary DESC        → thứ tự trong mỗi nhóm
```

Có từ MySQL 8.0 và SQLite 3.25.

In [7]:
import sqlite3

def window_demo(query):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Scores (id INTEGER, score REAL, dept INTEGER)")
    con.executemany("INSERT INTO Scores VALUES (?, ?, ?)",
                    [(1, 3.50, 1), (2, 3.65, 1), (3, 4.00, 2), (4, 3.85, 2), (5, 4.00, 1), (6, 3.65, 2)])
    cur = con.execute(query)
    print([d[0] for d in cur.description])
    for r in cur.fetchall():
        print("  ", r)
    con.close()

### Ba hàm đánh số: ROW_NUMBER, RANK, DENSE_RANK

Khác nhau ở cách xử lý **giá trị bằng nhau**:

| giá trị | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|
| 4.00 | 1 | 1 | 1 |
| 4.00 | 2 | 1 | 1 |
| 3.85 | 3 | **3** (nhảy) | **2** (liền) |

In [8]:
window_demo("""
SELECT score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
       RANK()       OVER (ORDER BY score DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense
FROM Scores ORDER BY score DESC
""")

['score', 'row_num', 'rnk', 'dense']
   (4.0, 1, 1, 1)
   (4.0, 2, 1, 1)
   (3.85, 3, 3, 2)
   (3.65, 4, 4, 3)
   (3.65, 5, 4, 3)
   (3.5, 6, 6, 4)


### PARTITION BY: đánh số riêng trong từng nhóm

"Top-k mỗi nhóm" = đánh số trong nhóm rồi lọc `<= k`. Vì `WHERE` chạy **trước** hàm cửa sổ, phải bọc trong truy vấn con hoặc `WITH`.

In [9]:
window_demo("""
WITH ranked AS (
    SELECT dept, id, score,
           DENSE_RANK() OVER (PARTITION BY dept ORDER BY score DESC) AS r
    FROM Scores
)
SELECT dept, id, score FROM ranked WHERE r = 1 ORDER BY dept, id
""")

['dept', 'id', 'score']
   (1, 5, 4.0)
   (2, 3, 4.0)


### LAG / LEAD: nhìn hàng trước / sau

`LAG(col, k)` lấy giá trị của hàng cách **trước** `k` hàng (theo `ORDER BY` của cửa sổ), `LEAD` lấy hàng **sau**; không có thì `NULL`.

In [10]:
window_demo("""
SELECT id, score,
       LAG(score, 1)  OVER (ORDER BY id) AS truoc,
       LEAD(score, 1) OVER (ORDER BY id) AS sau
FROM Scores ORDER BY id
""")

['id', 'score', 'truoc', 'sau']
   (1, 3.5, None, 3.65)
   (2, 3.65, 3.5, 4.0)
   (3, 4.0, 3.65, 3.85)
   (4, 3.85, 4.0, 4.0)
   (5, 4.0, 3.85, 3.65)
   (6, 3.65, 4.0, None)


### WITH (CTE): đặt tên cho bảng tạm

`WITH ten AS (SELECT ...) SELECT ... FROM ten` giúp chia truy vấn dài thành các bước dễ đọc — tương tự gán biến trung gian trong Python.

### Lỗi hay gặp

- Dùng `RANK` khi đề muốn thứ hạng **liền nhau** (cần `DENSE_RANK`), hoặc ngược lại.
- Lọc kết quả hàm cửa sổ trong `WHERE` cùng câu → lỗi; phải bọc truy vấn con / CTE.
- `rank` là **từ khóa** trong MySQL 8 → đặt tên cột là `` `rank` `` (backtick).
- Quên `PARTITION BY` → đánh số trên **toàn bảng** thay vì từng nhóm.

## 2.3 Pandas DataFrame — SQL bằng Python

### DataFrame là gì?

`pandas.DataFrame` là một **bảng** trong bộ nhớ: các cột có tên, mỗi cột là một `Series` (mảng có nhãn). LeetCode có phiên bản Pandas cho các bài SQL; mỗi thao tác SQL có một tương đương:

| SQL | Pandas |
|---|---|
| `WHERE cond` | `df[df["col"] > x]` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `DISTINCT` | `df.drop_duplicates()` / `series.unique()` |
| `GROUP BY` + gộp | `df.groupby("k")["v"].max()` |
| `JOIN` | `df1.merge(df2, left_on=..., right_on=...)` |
| `DENSE_RANK()` | `series.rank(method="dense", ascending=False)` |
| `LAG(col)` | `series.shift(1)` |

In [11]:
import pandas as pd

emp = pd.DataFrame({"id": [1, 2, 3, 4, 5],
                    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
                    "salary": [70000, 90000, 80000, 60000, 90000],
                    "departmentId": [1, 1, 2, 2, 1]})
dep = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
print(emp)

   id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


### Lọc, chọn cột, sắp xếp, bỏ trùng

In [12]:
print(emp[emp["salary"] > 65000][["name", "salary"]].sort_values("salary", ascending=False).head(2))
print("lương khác nhau giảm dần:", sorted(emp["salary"].unique().tolist(), reverse=True))

  name  salary
1  Jim   90000
4  Max   90000
lương khác nhau giảm dần: [90000, 80000, 70000, 60000]


### groupby và transform

- `groupby(k)[v].max()` → **co** mỗi nhóm thành một giá trị (như `GROUP BY`).
- `groupby(k)[v].transform("max")` → trả về giá trị của nhóm **cho từng hàng** (giống hàm cửa sổ `MAX() OVER (PARTITION BY k)`), rất tiện để lọc "hàng có giá trị lớn nhất nhóm".

In [13]:
print(emp.groupby("departmentId")["salary"].max())
top = emp[emp["salary"] == emp.groupby("departmentId")["salary"].transform("max")]
print(top)

departmentId
1    90000
2    80000
Name: salary, dtype: int64
   id   name  salary  departmentId
1   2    Jim   90000             1
2   3  Henry   80000             2
4   5    Max   90000             1


### merge (JOIN), rank, shift

In [14]:
m = emp.merge(dep, left_on="departmentId", right_on="id", suffixes=("", "_dep"))
print(m[["name_dep", "name", "salary"]])
s = pd.Series([4.00, 4.00, 3.85, 3.65])
print("rank dense:", s.rank(method="dense", ascending=False).tolist(), "| rank min (như RANK):", s.rank(method="min", ascending=False).tolist())
print("shift(1) (như LAG):", s.shift(1).tolist())

  name_dep   name  salary
0       IT    Joe   70000
1       IT    Jim   90000
2    Sales  Henry   80000
3    Sales    Sam   60000
4       IT    Max   90000
rank dense: [1.0, 1.0, 2.0, 3.0] | rank min (như RANK): [1.0, 1.0, 3.0, 4.0]
shift(1) (như LAG): [nan, 4.0, 4.0, 3.85]


### Lỗi hay gặp

- `df[...]` lọc trả về DataFrame **có thể rỗng** — kiểm tra `len(...)` trước khi lấy `.iloc[0]`.
- `rank` trả về số thực (`1.0`) → `astype(int)` nếu cần.
- `merge` hai bảng cùng tên cột → cột bị thêm hậu tố `_x`, `_y` (hoặc `suffixes` tự đặt).
- Khi đề yêu cầu trả `null`, trả `None` trong một DataFrame một hàng — không trả DataFrame rỗng.

## 2.4 Áp dụng vào bài này

"Ba hàng **liên tiếp** có cùng giá trị" cần so một hàng với hàng **trước/sau** nó: dùng **self-join** ba bản sao bảng với `id`, `id + 1`, `id + 2`, hoặc hàm cửa sổ **`LAG`/`LEAD`**. Dùng `DISTINCT` vì một số có thể xuất hiện liên tiếp nhiều lần. Bản Pandas: `shift(1)`, `shift(2)`.

# 3. Nền tảng

In [15]:
import sqlite3
import random
import pandas as pd


def run_sql(sql: str, tables: dict, params=()) -> pd.DataFrame:
    conn = sqlite3.connect(":memory:")
    for name, df in tables.items():
        df.to_sql(name, conn, index=False)
    out = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return out


def rows(df: pd.DataFrame) -> list:
    return [[None if pd.isna(v) else v for v in r] for r in df.itertuples(index=False)]


def logs_df(nums, ids=None) -> pd.DataFrame:
    ids = list(range(1, len(nums) + 1)) if ids is None else ids
    return pd.DataFrame({"id": ids, "num": [str(x) for x in nums]})   # num là varchar trong đề


EXAMPLE = logs_df([1, 1, 1, 2, 1, 2, 2])
print(EXAMPLE.T)

     0  1  2  3  4  5  6
id   1  2  3  4  5  6  7
num  1  1  1  2  1  2  2


## Tự nối bảng: dùng **bí danh** để coi một bảng như hai, ba bảng

`FROM Logs l1 JOIN Logs l2 ON l2.id = l1.id + 1` ghép mỗi dòng `l1` với dòng **ngay sau nó**. Bí danh (`l1`, `l2`, `l3`) là bắt buộc để phân biệt các "bản sao".

In [16]:
print(run_sql("""
SELECT l1.id AS id1, l1.num AS n1, l2.id AS id2, l2.num AS n2
FROM Logs l1
JOIN Logs l2 ON l2.id = l1.id + 1
""", {"Logs": EXAMPLE}))

   id1 n1  id2 n2
0    1  1    2  1
1    2  1    3  1
2    3  1    4  2
3    4  2    5  1
4    5  1    6  2
5    6  2    7  2


# 4. Cách 1: nối ba lần

Ba dòng liên tiếp `id, id+1, id+2` có cùng `num`:

```sql
SELECT DISTINCT l1.num AS ConsecutiveNums
FROM Logs l1
JOIN Logs l2 ON l2.id = l1.id + 1
JOIN Logs l3 ON l3.id = l1.id + 2
WHERE l1.num = l2.num AND l2.num = l3.num;
```

`DISTINCT` cần thiết: một đoạn `1,1,1,1` (bốn lần) tạo **hai** bộ ba thỏa → số `1` xuất hiện hai lần nếu không bỏ trùng.

In [17]:
SQL_JOIN = """
SELECT DISTINCT l1.num AS ConsecutiveNums
FROM Logs l1
JOIN Logs l2 ON l2.id = l1.id + 1
JOIN Logs l3 ON l3.id = l1.id + 2
WHERE l1.num = l2.num AND l2.num = l3.num
"""
print(run_sql(SQL_JOIN, {"Logs": EXAMPLE}))

  ConsecutiveNums
0               1

# 5. Cách 2: `LAG` / `LEAD`

`LAG(cột, k) OVER (ORDER BY id)` lấy giá trị của dòng **đứng trước** `k` vị trí; `LEAD` lấy dòng **đứng sau**. Không có dòng đó thì trả `NULL`.

In [18]:
print(run_sql("""
SELECT id, num,
       LAG(num, 1)  OVER (ORDER BY id) AS prev1,
       LEAD(num, 1) OVER (ORDER BY id) AS next1,
       LEAD(num, 2) OVER (ORDER BY id) AS next2
FROM Logs
""", {"Logs": EXAMPLE}))

   id num prev1 next1 next2
0   1   1   NaN     1     1
1   2   1     1     1     2
2   3   1     1     2     1
3   4   2     1     1     2
4   5   1     2     2     2
5   6   2     1     2   NaN
6   7   2     2   NaN   NaN


In [19]:
SQL_LEAD = """
SELECT DISTINCT num AS ConsecutiveNums
FROM (
    SELECT num,
           LEAD(num, 1) OVER (ORDER BY id) AS n1,
           LEAD(num, 2) OVER (ORDER BY id) AS n2
    FROM Logs
) AS t
WHERE num = n1 AND num = n2
"""
print(run_sql(SQL_LEAD, {"Logs": EXAMPLE}))

  ConsecutiveNums
0               1


Không thể viết `WHERE LEAD(num) OVER (...) = num` trực tiếp: hàm cửa sổ được tính **sau** `WHERE` (thứ tự logic FROM → WHERE → … → SELECT). Vì vậy phải tính trong truy vấn con rồi lọc ở ngoài.

# 6. Cách 3: "đảo liên tiếp" (gaps and islands)

Hai cách trên giả định `id` liền nhau (1, 2, 3…). Nếu `id` có **lỗ** (ví dụ đã xóa dòng), "liên tiếp" nên hiểu là liên tiếp theo **thứ tự** `id`. Kỹ thuật tổng quát:

- `rn_all` = `ROW_NUMBER() OVER (ORDER BY id)` — số thứ tự trong cả bảng.
- `rn_num` = `ROW_NUMBER() OVER (PARTITION BY num ORDER BY id)` — số thứ tự **trong nhóm cùng số**.
- Trong một đoạn liên tiếp cùng `num`, cả hai cùng tăng 1 mỗi dòng → **hiệu `rn_all − rn_num` không đổi**. Sang đoạn khác (bị số khác chen vào) thì hiệu tăng.

```text
id   num  rn_all  rn_num  hiệu
1    1    1       1       0   ┐
2    1    2       2       0   │ đảo (num=1, hiệu 0): 3 dòng ✓
3    1    3       3       0   ┘
4    2    4       1       3     đảo (num=2, hiệu 3): 1 dòng
5    1    5       4       1     đảo (num=1, hiệu 1): 1 dòng
6    2    6       2       4   ┐ đảo (num=2, hiệu 4): 2 dòng
7    2    7       3       4   ┘
```

Nhóm theo `(num, hiệu)`, giữ các nhóm có `COUNT(*) >= 3`.

In [20]:
print(run_sql("""
SELECT id, num,
       ROW_NUMBER() OVER (ORDER BY id) AS rn_all,
       ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS rn_num,
       ROW_NUMBER() OVER (ORDER BY id) - ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS grp
FROM Logs
ORDER BY id
""", {"Logs": EXAMPLE}))

   id num  rn_all  rn_num  grp
0   1   1       1       1    0
1   2   1       2       2    0
2   3   1       3       3    0
3   4   2       4       1    3
4   5   1       5       4    1
5   6   2       6       2    4
6   7   2       7       3    4


In [21]:
SQL_ISLAND = """
SELECT DISTINCT num AS ConsecutiveNums
FROM (
    SELECT num,
           ROW_NUMBER() OVER (ORDER BY id) - ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS grp
    FROM Logs
) AS t
GROUP BY num, grp
HAVING COUNT(*) >= 3
"""
print(run_sql(SQL_ISLAND, {"Logs": EXAMPLE}))

  ConsecutiveNums
0               1


# 7. Mô phỏng bằng Python

In [22]:
def trace_runs(nums):
    runs, i = [], 0
    while i < len(nums):
        j = i
        while j < len(nums) and nums[j] == nums[i]:
            j += 1
        runs.append((nums[i], j - i))
        i = j
    print("các đoạn liên tiếp (số, độ dài):", runs)
    ans = sorted({str(v) for v, L in runs if L >= 3})
    print("→ số có đoạn dài >= 3:", ans)
    return ans

_ = trace_runs([1, 1, 1, 2, 1, 2, 2])
_ = trace_runs([3, 3, 3, 3, 5, 5, 5, 3])

các đoạn liên tiếp (số, độ dài):

 [(1, 3), (2, 1), (1, 1), (2, 2)]
→ số có đoạn dài >= 3: ['1']
các đoạn liên tiếp (số, độ dài): [(3, 4), (5, 3), (3, 1)]
→ số có đoạn dài >= 3: ['3', '5']


# 8. Phiên bản nộp LeetCode

**MySQL** — cách 1 (mục 4) hoặc cách 2 (mục 5, MySQL 8+).

**Pandas:** dùng `shift` (giống `LAG/LEAD`) trên bảng đã sắp theo `id`.

In [23]:
def consecutive_numbers(logs: pd.DataFrame) -> pd.DataFrame:
    s = logs.sort_values("id")["num"].reset_index(drop=True)
    mask = (s == s.shift(-1)) & (s == s.shift(-2))
    return pd.DataFrame({"ConsecutiveNums": s[mask].drop_duplicates().tolist()})

print(consecutive_numbers(EXAMPLE))

  ConsecutiveNums
0               1

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


def as_set(df):
    return sorted(r[0] for r in rows(df))


def expected_consecutive(nums):
    ans, i = set(), 0
    while i < len(nums):
        j = i
        while j < len(nums) and nums[j] == nums[i]:
            j += 1
        if j - i >= 3:
            ans.add(str(nums[i]))
        i = j
    return sorted(ans)


T = {"Logs": EXAMPLE}
check("Ví dụ (JOIN)", as_set(run_sql(SQL_JOIN, T)), ["1"])
check("Ví dụ (LEAD)", as_set(run_sql(SQL_LEAD, T)), ["1"])
check("Ví dụ (đảo)", as_set(run_sql(SQL_ISLAND, T)), ["1"])
check("Ví dụ (Pandas)", as_set(consecutive_numbers(EXAMPLE)), ["1"])
check("Bốn lần (DISTINCT)", rows(run_sql(SQL_JOIN, {"Logs": logs_df([7, 7, 7, 7])})), [["7"]])
check("Không có", as_set(run_sql(SQL_JOIN, {"Logs": logs_df([1, 2, 1, 2])})), [])

random.seed(180)
for _ in range(300):
    nums = [random.randint(1, 3) for _ in range(random.randint(1, 12))]
    exp = expected_consecutive(nums)
    t = {"Logs": logs_df(nums)}
    got = [as_set(run_sql(q, t)) for q in (SQL_JOIN, SQL_LEAD, SQL_ISLAND)] + [as_set(consecutive_numbers(logs_df(nums)))]
    assert all(g == exp for g in got), (nums, got, exp)
print("300 bảng ngẫu nhiên: ba câu SQL và bản Pandas khớp")

Ví dụ (JOIN)         PASS  ['1']
Ví dụ (LEAD)         PASS  ['1']
Ví dụ (đảo)          PASS  ['1']
Ví dụ (Pandas)       PASS  ['1']


Bốn lần (DISTINCT)   PASS  [['7']]
Không có             PASS  []


300 bảng ngẫu nhiên: ba câu SQL và bản Pandas khớp


# 9. Vì sao đúng? Độ phức tạp?

- **Nối ba lần:** với `id` liền nhau, `(l1, l2, l3)` duyệt đúng mọi bộ ba dòng liên tiếp; điều kiện bằng nhau chọn bộ ba cùng số. Một số có đoạn dài `>= 3` ⟺ tồn tại bộ ba như vậy.
- **LEAD:** giống hệt nhưng "dòng sau" được xác định theo **thứ tự** `id`, nên đúng cả khi `id` có lỗ.
- **Đảo:** hiệu `rn_all − rn_num` không đổi trong một đoạn liên tiếp cùng số, và khác nhau giữa các đoạn khác nhau của cùng một số (giữa hai đoạn có ít nhất một dòng số khác làm `rn_all` tăng thêm mà `rn_num` không tăng).

| Cách | Độ phức tạp | Đúng khi `id` có lỗ? |
|---|---|---|
| Nối ba lần (có chỉ mục trên `id`) | `O(n)` – `O(n log n)` | không |
| `LEAD` | `O(n log n)` | có |
| Đảo liên tiếp | `O(n log n)` | có, và đổi `3` thành `k` bất kỳ |

# 10. Những lỗi dễ mắc

## Lỗi 1: Quên `DISTINCT`

In [25]:
SQL_NO_DISTINCT = SQL_JOIN.replace("SELECT DISTINCT", "SELECT")
t4 = {"Logs": logs_df([7, 7, 7, 7, 7])}
print("Sai :", rows(run_sql(SQL_NO_DISTINCT, t4)), "| Đúng:", rows(run_sql(SQL_JOIN, t4)))

Sai : [['7'], ['7'], ['7']] | Đúng: [['7']]


## Lỗi 2: Đếm tổng số lần xuất hiện thay vì số lần **liên tiếp**

In [26]:
SQL_COUNT_ALL = """
SELECT num AS ConsecutiveNums FROM Logs GROUP BY num HAVING COUNT(*) >= 3
"""
t5 = {"Logs": logs_df([1, 2, 1, 2, 1, 2])}
print("Sai :", as_set(run_sql(SQL_COUNT_ALL, t5)), "| Đúng:", as_set(run_sql(SQL_JOIN, t5)))

Sai : ['1', '2'] | Đúng: []


## Lỗi 3: Nối theo `id + 1` khi `id` có lỗ

In [27]:
gappy = logs_df([4, 4, 4], ids=[1, 2, 5])           # dòng id 3, 4 đã bị xóa
print("JOIN theo id+1 :", as_set(run_sql(SQL_JOIN, {"Logs": gappy})))
print("LEAD (thứ tự)  :", as_set(run_sql(SQL_LEAD, {"Logs": gappy})))

JOIN theo id+1 : []
LEAD (thứ tự)  : ['4']


Đề này đảm bảo `id` tự tăng liền nhau nên cách nối vẫn được chấp nhận; nhưng hãy nhớ hai cách hiểu "liên tiếp" (theo giá trị `id` hay theo **thứ tự**) cho kết quả khác nhau khi có lỗ.

## Lỗi 4: Dùng hàm cửa sổ trong `WHERE`

In [28]:
try:
    run_sql("SELECT num FROM Logs WHERE num = LEAD(num) OVER (ORDER BY id)", {"Logs": EXAMPLE})
except Exception as e:
    print(type(e).__name__, "→", str(e).split(":")[-1].strip())

DatabaseError → misuse of window function LEAD()


# 11. Mẹo nhận diện pattern

- **So một dòng với dòng trước/sau** → `LAG`/`LEAD` (hoặc self-join theo khóa liền kề).
- **Đoạn liên tiếp cùng giá trị / chuỗi ngày liên tiếp** → "gaps and islands": `ROW_NUMBER` toàn bảng trừ `ROW_NUMBER` trong nhóm (hoặc trừ chính giá trị ngày).
- **Mỗi giá trị chỉ một lần trong kết quả** → `DISTINCT`.

Bài liên quan: **#601 Human Traffic of Stadium** (Hard), **#1225 Report Contiguous Dates** (Hard), **#1285 Find the Start and End Number of Continuous Ranges**, **#197 Rising Temperature** (Easy, so với ngày trước).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `num = [5, 5, 6, 6, 6, 5, 5, 5, 5]`, lập bảng `rn_all`, `rn_num`, hiệu, và kết quả.

## Bài 2 — Viết câu SQL cách nối ba lần

In [29]:
def exercise_2_sql() -> str:
    # TODO
    pass


# assert sorted(r[0] for r in rows(run_sql(exercise_2_sql(), {"Logs": logs_df([1, 1, 1, 2, 1, 2, 2])}))) == ["1"]
# assert rows(run_sql(exercise_2_sql(), {"Logs": logs_df([9, 9, 9, 9])})) == [["9"]]

## Bài 3 — Viết câu SQL cách `LAG` (nhìn **lùi** hai dòng thay vì tiến)

In [30]:
def exercise_3_sql() -> str:
    # TODO
    pass


# assert sorted(r[0] for r in rows(run_sql(exercise_3_sql(), {"Logs": logs_df([2, 2, 2, 3, 3, 3, 1])}))) == ["2", "3"]
# assert rows(run_sql(exercise_3_sql(), {"Logs": logs_df([1, 1])})) == []

## Bài 4 — Biến thể: liệt kê **mọi đoạn** dài ít nhất `k` (tham số `?`)

Trả về `num, start_id, end_id, len` của mỗi đoạn liên tiếp dài `>= k`, sắp theo `start_id`.

In [31]:
def exercise_4_sql() -> str:
    # TODO: câu SQL có một tham số ? là k
    pass


# assert rows(run_sql(exercise_4_sql(), {"Logs": logs_df([5, 5, 6, 6, 6, 5, 5, 5, 5])}, params=(3,))) == [["6", 3, 5, 3], ["5", 6, 9, 4]]
# assert rows(run_sql(exercise_4_sql(), {"Logs": logs_df([1, 2, 3])}, params=(2,))) == []

## Bài 5 — Mini challenge: bản Pandas "đảo liên tiếp" cho `k` bất kỳ

Gợi ý: mã đoạn = `(s != s.shift()).cumsum()` (tăng 1 mỗi khi giá trị đổi); đếm độ dài từng đoạn bằng `groupby`.

In [32]:
def exercise_5_consecutive_k(logs: pd.DataFrame, k: int) -> pd.DataFrame:
    # TODO: trả về DataFrame cột ConsecutiveNums (mỗi số một lần, sắp tăng dần theo chuỗi)
    pass


# assert exercise_5_consecutive_k(logs_df([1, 1, 1, 2, 1, 2, 2]), 3)["ConsecutiveNums"].tolist() == ["1"]
# assert exercise_5_consecutive_k(logs_df([1, 1, 1, 2, 1, 2, 2]), 2)["ConsecutiveNums"].tolist() == ["1", "2"]
# assert exercise_5_consecutive_k(logs_df([4, 5]), 2)["ConsecutiveNums"].tolist() == []

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy câu SQL ở mục 6 (bảng `grp`) với bảng mới sau khi tự làm.

**Bài 3**
- Mức 1: `LAG(num, 1)` và `LAG(num, 2)` trong truy vấn con; lọc `num = p1 AND num = p2`.

**Bài 4**
- Mức 1: như `SQL_ISLAND`, nhưng chọn thêm `MIN(id)`, `MAX(id)`, `COUNT(*)`; `HAVING COUNT(*) >= ?`.

**Bài 5**
- Mức 1: `s = logs.sort_values("id")["num"]`; `g = (s != s.shift()).cumsum()`.
- Mức 2: `sizes = s.groupby(g).agg(["first", "size"])`; lọc `size >= k`, lấy `first`, bỏ trùng, sắp xếp.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [33]:
print(run_sql("""
SELECT id, num,
       ROW_NUMBER() OVER (ORDER BY id) AS rn_all,
       ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS rn_num,
       ROW_NUMBER() OVER (ORDER BY id) - ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS grp
FROM Logs ORDER BY id
""", {"Logs": logs_df([5, 5, 6, 6, 6, 5, 5, 5, 5])}))
print("→", as_set(run_sql(SQL_ISLAND, {"Logs": logs_df([5, 5, 6, 6, 6, 5, 5, 5, 5])})))

   id num  rn_all  rn_num  grp
0   1   5       1       1    0
1   2   5       2       2    0
2   3   6       3       1    2
3   4   6       4       2    2
4   5   6       5       3    2
5   6   5       6       3    3
6   7   5       7       4    3
7   8   5       8       5    3
8   9   5       9       6    3
→ ['5', '6']


## Đáp án Bài 2

In [34]:
def answer_2_sql() -> str:
    return SQL_JOIN

assert sorted(r[0] for r in rows(run_sql(answer_2_sql(), {"Logs": logs_df([1, 1, 1, 2, 1, 2, 2])}))) == ["1"]
assert rows(run_sql(answer_2_sql(), {"Logs": logs_df([9, 9, 9, 9])})) == [["9"]]
print("Bài 2 PASS (giống SQL_JOIN ở mục 4)")

Bài 2 PASS (giống SQL_JOIN ở mục 4)


## Đáp án Bài 3

In [35]:
def answer_3_sql() -> str:
    return """
    SELECT DISTINCT num AS ConsecutiveNums
    FROM (
        SELECT num,
               LAG(num, 1) OVER (ORDER BY id) AS p1,
               LAG(num, 2) OVER (ORDER BY id) AS p2
        FROM Logs
    ) AS t
    WHERE num = p1 AND num = p2
    """

assert sorted(r[0] for r in rows(run_sql(answer_3_sql(), {"Logs": logs_df([2, 2, 2, 3, 3, 3, 1])}))) == ["2", "3"]
assert rows(run_sql(answer_3_sql(), {"Logs": logs_df([1, 1])})) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [36]:
def answer_4_sql() -> str:
    return """
    SELECT num, MIN(id) AS start_id, MAX(id) AS end_id, COUNT(*) AS len
    FROM (
        SELECT id, num,
               ROW_NUMBER() OVER (ORDER BY id) - ROW_NUMBER() OVER (PARTITION BY num ORDER BY id) AS grp
        FROM Logs
    ) AS t
    GROUP BY num, grp
    HAVING COUNT(*) >= ?
    ORDER BY start_id
    """

assert rows(run_sql(answer_4_sql(), {"Logs": logs_df([5, 5, 6, 6, 6, 5, 5, 5, 5])}, params=(3,))) == [["6", 3, 5, 3], ["5", 6, 9, 4]]
assert rows(run_sql(answer_4_sql(), {"Logs": logs_df([1, 2, 3])}, params=(2,))) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [37]:
def answer_5_consecutive_k(logs: pd.DataFrame, k: int) -> pd.DataFrame:
    s = logs.sort_values("id")["num"].reset_index(drop=True)
    g = (s != s.shift()).cumsum()
    runs = s.groupby(g).agg(["first", "size"])
    vals = sorted(set(runs.loc[runs["size"] >= k, "first"]))
    return pd.DataFrame({"ConsecutiveNums": vals})

assert answer_5_consecutive_k(logs_df([1, 1, 1, 2, 1, 2, 2]), 3)["ConsecutiveNums"].tolist() == ["1"]
assert answer_5_consecutive_k(logs_df([1, 1, 1, 2, 1, 2, 2]), 2)["ConsecutiveNums"].tolist() == ["1", "2"]
assert answer_5_consecutive_k(logs_df([4, 5]), 2)["ConsecutiveNums"].tolist() == []
for _ in range(300):
    nums = [random.randint(1, 3) for _ in range(random.randint(1, 12))]
    assert answer_5_consecutive_k(logs_df(nums), 3)["ConsecutiveNums"].tolist() == expected_consecutive(nums)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Ba dòng liên tiếp cùng số: nối `Logs l1, l2, l3` theo `id`, `id+1`, `id+2` (hoặc `LEAD(num, 1)`, `LEAD(num, 2)` trong truy vấn con), lọc bằng nhau, `SELECT DISTINCT`. Tổng quát cho `k` bất kỳ: "đảo liên tiếp" — nhóm theo `num` và `ROW_NUMBER() − ROW_NUMBER() OVER (PARTITION BY num)`, `HAVING COUNT(*) >= k`.

---

## Checklist trước khi nộp

- [ ] Có `DISTINCT`?
- [ ] Đếm **liên tiếp**, không phải tổng số lần?
- [ ] Hàm cửa sổ tính trong truy vấn con, lọc ở ngoài?
- [ ] Tên cột `ConsecutiveNums`?